# cell2location — Methodology & Architecture

**Paper:** Kleshchevnikov, Shmatko, Dann et al., *Cell2location maps fine-grained cell types in spatial transcriptomics*, Nature Biotechnology 40, 661–671 (2022)
**Code:** [BayraktarLab/cell2location](https://github.com/BayraktarLab/cell2location)
**Companion notebook:** `cell2location_tutorial.ipynb` (this folder) — the executable run
**This notebook:** a methodology / architecture reference — no code execution, markdown only

---

This document walks through *how* cell2location works internally: the biological problem it addresses, the two-stage Bayesian model architecture, the mathematical intuition behind each stage, the data flow from raw inputs to final spatial maps, and how to interpret the output.

## 1. The Problem

Spatial transcriptomics technologies (e.g. 10x Genomics Visium) measure gene expression while preserving tissue location — but each measurement "spot" is physically larger than a single cell, and therefore captures RNA from **multiple cells mixed together**.

Separately, single-cell RNA-seq (scRNA-seq) gives a clean, per-cell-type expression signature — but with **no spatial information**, since the tissue is dissociated before sequencing.

**Goal:** combine both data sources to estimate the abundance of each cell type at every spatial location — i.e., deconvolve the per-spot mixture back into its cell-type components.

## 2. High-Level Architecture

```mermaid
flowchart TD
    A[scRNA-seq reference<br/>labelled cell types] --> B[Stage 1:<br/>Regression Model<br/>Negative Binomial]
    B --> C[Reference signatures<br/>inf_aver: genes x cell types]
    D[Spatial transcriptomics data<br/>Visium spots] --> E[Stage 2:<br/>Cell2location Model<br/>Bayesian deconvolution]
    C --> E
    E --> F[Posterior distribution<br/>per spot, per cell type]
    F --> G[5% quantile<br/>confident abundance]
    F --> H[Posterior mean<br/>point estimate]
    G --> I[Spatial abundance maps]
    H --> I
    I --> J[Downstream analysis:<br/>tissue region clustering]
```

Two models are trained in sequence:
1. **Regression Model** (Stage 1) — learns per-cell-type expression signatures from labelled scRNA-seq data.
2. **Cell2location Model** (Stage 2) — uses those signatures as a fixed reference to decompose spatial spot counts into cell-type abundances.

## 3. Stage 1 — Reference Cell Type Signature Estimation

**Input:** scRNA-seq count matrix (cells × genes), with each cell labelled by cell type/subset and batch/sample of origin.

**Model:** a Negative Binomial (NB) regression model. RNA count data is over-dispersed (high variance relative to the mean, many near-zero values), so a plain Poisson or Gaussian model would fit poorly — NB is the standard choice for this kind of count data in genomics.

**What it learns:** for every gene, the *average expression level characteristic of each cell type*, while separately accounting for:
- **Batch/sample effects** — systematic differences between different scRNA-seq experiments contributing to the reference
- **Technical variation** — sequencing depth differences, dropout

**Output:** a matrix `inf_aver` of shape `[genes × cell types]` — this is the reference cell type signature matrix, representing "if a spot contained *only* cell type X, this is roughly the expression profile you'd expect."

**Training:** optimized via variational inference (an approximate Bayesian fitting method), tracked through an ELBO (Evidence Lower Bound) loss curve that should decrease and plateau as training converges.

## 4. Stage 2 — Spatial Mapping

**Input:**
- Spatial transcriptomics count matrix (spots × genes)
- Reference signature matrix `inf_aver` from Stage 1 (subset to genes shared between both datasets)

**Model:** the core cell2location generative model. Conceptually, it assumes each spot's observed gene counts arise from a **weighted combination of the reference cell type signatures**, where the weights represent how many cells of each type are present — plus explicit terms for technical noise sources:

- **Cell type abundance** `w` — the main quantity of interest: how many cells of each type at each spot
- **Platform/technology scaling factor** — corrects for systematic differences in sensitivity between the reference and spatial technologies
- **Contaminating RNA term** — accounts for RNA that diffused between neighboring spots rather than being locally produced
- **Per-location detection efficiency** — accounts for spot-to-spot technical variability in total RNA capture

**User-set hyperparameters:**
| Parameter | Meaning | Typical value |
|---|---|---|
| `N_cells_per_location` | Expected number of cells per spot, ideally estimated from paired histology | tissue-dependent (e.g. 30 for lymph node) |
| `detection_alpha` | Regularization strength on per-location technical normalization | 200 (low technical variability) or 20 (high technical variability) |

**Inference:** the model is fit using stochastic variational inference on the full dataset (`batch_size=None`, `train_size=1` — all spots used every epoch, since abundance needs to be estimated everywhere, not just on a training subset). This stage typically requires many more training epochs than Stage 1 (tutorial default: 30,000) since it's solving a substantially higher-dimensional problem — abundance of every cell type at every spot simultaneously.

**Output:** a full posterior distribution per spot, per cell type — not a single number, but a distribution capturing the model's uncertainty. In practice summarized as:
- The **5% quantile** ("at least this amount is present") — conservative, high-confidence estimate
- The **posterior mean** — best single-point estimate

## 5. End-to-End Data Flow

```mermaid
sequenceDiagram
    participant SC as scRNA-seq reference
    participant R as RegressionModel (Stage 1)
    participant SIG as inf_aver (signatures)
    participant SP as Visium spatial data
    participant C2L as Cell2location Model (Stage 2)
    participant OUT as adata_vis.obsm / .obs

    SC->>R: train (max_epochs=250)
    R->>SIG: export_estimated_expression()
    SP->>C2L: setup_anndata + shared genes
    SIG->>C2L: cell_state_df=inf_aver
    C2L->>C2L: train (max_epochs=10000-30000)
    C2L->>OUT: export_posterior() -> q05_cell_abundance_w_sf
    OUT->>OUT: sc.pl.spatial() -> abundance maps
```

## 6. Quality Control Checkpoints

The pipeline includes diagnostics at each stage to verify the model actually fit the data well, rather than trusting output blindly:

| Checkpoint | What it checks | Healthy sign |
|---|---|---|
| `mod.plot_history()` (Stage 1 & 2) | ELBO loss curve over training | Decreasing trend that plateaus by the end |
| `mod.plot_QC()` (Stage 1) | Reconstruction accuracy — predicted vs. observed expression | Points cluster along the diagonal |
| Biological plausibility (Stage 2) | Do co-located cell types match known tissue anatomy? | e.g. germinal-center-associated types co-localize; T-zone / B-zone segregation is recovered |

The last checkpoint — biological plausibility — is arguably the strongest validation: the model is never told anatomical structure in advance, so recovering known tissue organization from RNA counts alone is meaningful external validation.

## 7. Interpreting the Output

**Per-cell-type spatial grid** — each panel shows one cell type's predicted abundance across the tissue as a heatmap overlaid on the histology image. Brightness = higher confident abundance.

**Composite overlay** — multiple cell types shown as different colors on the same tissue image, to visualize spatial relationships (overlap vs. segregation) between cell types directly.

**Downstream (optional):** cell abundance profiles can be clustered (K-nearest-neighbours + Leiden clustering) to automatically discover discrete tissue regions with similar cell-type composition — effectively data-driven anatomical zonation.

## 8. Summary

| Stage | Model | Input | Output | Approx. runtime (T4 GPU) |
|---|---|---|---|---|
| 1 | Negative Binomial Regression | scRNA-seq reference (labelled) | Cell type signature matrix | ~15-20 min |
| 2 | Cell2location (Bayesian deconvolution) | Spatial data + Stage 1 signatures | Per-spot, per-cell-type abundance posterior | ~30 min - 2 hr (epoch-dependent) |

See `cell2location_tutorial.ipynb` in this folder for the executable version of this pipeline, with real code cells and generated output figures.